# Setup


In [ ]:
HADOOP_START_FROM_SCRATCH = True
HADOOP_VPN_DOMAIN = "mavasbel.vpn.itam.mx"
DOCKER_INTERNAL_HOST = "host.docker.internal"
DOCKER_DNS = ["10.15.20.1"]

HADOOP_NAMENODE_HOSTNAME = f"namenode.{HADOOP_VPN_DOMAIN}"
HADOOP_NAMENODE_IP = "10.15.20.2"
HADOOP_NAMENODE_PORT = 8020
HADOOP_NAMENODE_WEBUI_PORT = 9870

HADOOP_RESOURCEMANAGER_HOSTNAME = f"resourcemanager.{HADOOP_VPN_DOMAIN}"
HADOOP_RESOURCEMANAGER_IP = "10.15.20.2"
HADOOP_RESOURCEMANAGER_WEBUI_PORT = 8088
HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT = 8032
HADOOP_RESOURCEMANAGER_TRACKER_PORT = 8031
HADOOP_RESOURCEMANAGER_SCHEDULER_PORT = 8030
HADOOP_RESOURCEMANAGER_ADMIN_PORT = 8033

HADOOP_MAPRED_JOB_HISTORY_PORT = 10020
HADOOP_MAPRED_LOG_SERVER_PORT = 19888

HADOOP_REPLICATION = 3
HADOOP_NUM_WORKERS = 3

HADOOP_DATANODE_IPS = ["10.15.20.2"] * 3
HADOOP_DATANODE_NAMES = [f"datanode-{i+1}" for i in range(HADOOP_NUM_WORKERS)]
HADOOP_DATANODE_HOSTNAMES = [
    f"{HADOOP_DATANODE_NAMES[i]}.{HADOOP_VPN_DOMAIN}" for i in range(HADOOP_NUM_WORKERS)
]
HADOOP_DATANODE_WEBUI_PORTS = [9864 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]
HADOOP_DATANODE_TRANSFER_PORTS = [9866 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]
HADOOP_DATANODE_IPC_PORTS = [6867 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]

HADOOP_NODEMANAGER_IPS = ["10.15.20.2"] * 3
HADOOP_NODEMANAGER_NAMES = [f"nodemanager-{i+1}" for i in range(HADOOP_NUM_WORKERS)]
HADOOP_NODEMANAGER_HOSTNAMES = [
    f"{HADOOP_NODEMANAGER_NAMES[i]}.{HADOOP_VPN_DOMAIN}"
    for i in range(HADOOP_NUM_WORKERS)
]
HADOOP_NODEMANAGER_WEBUI_PORTS = [8050 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]
HADOOP_NODEMANAGER_RPC_PORTS = [8051 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]

HADOOP_WORKDIR = "/opt/hadoop/work-dir"
HADOOP_NAMENODE_NAMEDIR = "/opt/hadoop/dfs/name"
HADOOP_DATANODE_DATADIR = "/opt/hadoop/dfs/data"

HADOOP_HDFS_DATADIR = "/opt/hadoop/work-dir"

In [ ]:
import os
from pathlib import Path

LOCALHOST_WORKDIR = f"{os.path.join(os.path.relpath(Path.cwd()))}"
DOCKER_MOUNTDIR = os.path.join(LOCALHOST_WORKDIR, "mount")

Path(DOCKER_MOUNTDIR).mkdir(parents=True, exist_ok=True)

# Stop hadoop-cluster.docker-compose.yml


In [ ]:
!docker compose -f hadoop-cluster.docker-compose.yml down -v

In [ ]:
import shutil

if HADOOP_START_FROM_SCRATCH:
    shutil.rmtree(DOCKER_MOUNTDIR, ignore_errors=True)
    Path(DOCKER_MOUNTDIR).mkdir(parents=True, exist_ok=True)

# Start hadoop-cluster.docker-compose.yml


In [ ]:
import os
import yaml
from IPython.display import Markdown, display

# Configs can be valited in the running containers with
# hdfs getconf -confKey {key}
# or printing
# Four principal servicios
# cat $HADOOP_HOME/etc/hadoop/core-site.xml       # fs.defaultFS, trash, security, proxyuser
# cat $HADOOP_HOME/etc/hadoop/hdfs-site.xml       # replication, block size, permissions, namenode dirs
# cat $HADOOP_HOME/etc/hadoop/yarn-site.xml       # resourcemanager, nodemanager, memory
# cat $HADOOP_HOME/etc/hadoop/mapred-site.xml     # framework, classpath, JVM opts
# Daemons env vars
# cat $HADOOP_HOME/etc/hadoop/hadoop-env.sh       # JAVA_HOME, HADOOP_HEAPSIZE, HADOOP_LOG_DIR
# cat $HADOOP_HOME/etc/hadoop/yarn-env.sh         # YARN_HEAPSIZE, YARN_LOG_DIR
# cat $HADOOP_HOME/etc/hadoop/mapred-env.sh       # HADOOP_JOB_HISTORYSERVER_HEAPSIZE
# Workers / topology
# cat $HADOOP_HOME/etc/hadoop/workers             # DataNodes/NodeManagers
# cat $HADOOP_HOME/etc/hadoop/masters             # SecondaryNameNode
# Logging
# cat $HADOOP_HOME/etc/hadoop/log4j.properties    # Log levels per component
# Scheduler capacity
# cat $HADOOP_HOME/etc/hadoop/capacity-scheduler.xml
# All of them
# for f in $HADOOP_HOME/etc/hadoop/*.xml; do
#     echo ""; echo "════════ $f ════════"; cat $f
# done

env_content = {
    "HADOOP_HOME": "/opt/hadoop",
    "HADOOP_WORKDIR": HADOOP_WORKDIR,
    "HADOOP_HEAPSIZE_MAX": "768",
    "YARN_HEAPSIZE": "512",
    "MAPRED-SITE.XML_mapreduce.framework.name": "yarn",
    "MAPRED-SITE.XML_mapreduce.application.classpath": ":".join(
        [
            "/opt/hadoop/share/hadoop/mapreduce/*",
            "/opt/hadoop/share/hadoop/common/*",
            "/opt/hadoop/share/hadoop/common/lib/*",
            "/opt/hadoop/share/hadoop/hdfs/*",
            "/opt/hadoop/share/hadoop/hdfs/lib/*",
            "/opt/hadoop/share/hadoop/yarn/*",
            "/opt/hadoop/share/hadoop/yarn/lib/*",
        ]
    ),
    # "CORE-SITE.XML_dfs.replication": HADOOP_REPLICATION,
    "CORE-SITE.XML_fs.defaultFS": f"hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}",
    "CORE-SITE.XML_fs.trash.interval": "1440",  # Tiempo de vida de la papelera en minutos (1440 = 24h)
    "CORE-SITE.XML_fs.trash.checkpoint.interval": "60",  # Cada 60 minutos crea un nuevo checkpoint en .Trash
    "CORE-SITE.XML_hadoop.proxyuser.hive.hosts": "*",
    "CORE-SITE.XML_hadoop.proxyuser.hive.groups": "*",
    "CORE-SITE.XML_hadoop.security.authentication": "simple",  # Sin Kerberos
    "HDFS-SITE.XML_dfs.replication": HADOOP_REPLICATION,
    "HDFS-SITE.XML_dfs.permissions.enabled": "true",  # Aplica permisos POSIX
    "HDFS-SITE.XML_dfs.namenode.name.dir": HADOOP_NAMENODE_NAMEDIR,
    "HDFS-SITE.XML_dfs.namenode.name.dir.perm": "775",
    # "HDFS-SITE.XML_dfs.namenode.rpc-address": f"0.0.0.0:{HADOOP_NAMENODE_PORT}",
    # "HDFS-SITE.XML_dfs.namenode.http-address": f"0.0.0.0:{HADOOP_NAMENODE_WEBUI_PORT}",
    "HDFS-SITE.XML_dfs.datanode.use.datanode.hostname": "true",
    "HDFS-SITE.XML_dfs.datanode.data.dir": HADOOP_DATANODE_DATADIR,
    "HDFS_DATANODE_OPTS": "-Xmx1024m",  # Heap DataNode
    "YARN-SITE.XML_yarn.resourcemanager.hostname": HADOOP_RESOURCEMANAGER_HOSTNAME,
    # "YARN-SITE.XML_yarn.resourcemanager.webapp.address": f"0.0.0.0:{HADOOP_RESOURCEMANAGER_WEBUI_PORT}",
    # "YARN-SITE.XML_yarn.resourcemanager.address": f"0.0.0.0:{HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT}",
    # "YARN-SITE.XML_yarn.resourcemanager.scheduler.address": f"0.0.0.0:{HADOOP_RESOURCEMANAGER_SCHEDULER_PORT}",
    # "YARN-SITE.XML_yarn.resourcemanager.resource-tracker.address": f"0.0.0.0:{HADOOP_RESOURCEMANAGER_TRACKER_PORT}",
    # "YARN-SITE.XML_yarn.resourcemanager.admin.address": f"0.0.0.0:{HADOOP_RESOURCEMANAGER_ADMIN_PORT}",
    "YARN-SITE.XML_yarn.nodemanager.aux-services": "mapreduce_shuffle",
    "YARN-SITE.XML_yarn.nodemanager.vmem-check-enabled": "false",
    "YARN-SITE.XML_yarn.nodemanager.resource.memory-mb": "1792",  # Memoria total que YARN puede repartir en un nodo
    "YARN-SITE.XML_yarn.scheduler.maximum-allocation-mb": "1792",  # Tamaño máximo de un solo contenedor
    "YARN-SITE.XML_yarn.scheduler.minimum-allocation-mb": "128",  # Tamaño mínimo de asignación
    "YARN-SITE.XML_yarn.log-aggregation-enable": "true",
    "YARN-SITE.XML_yarn.log.server.url": f"http://{HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_MAPRED_LOG_SERVER_PORT}/jobhistory/logs",  # URL para que la Web UI de YARN sepa a dónde redirigir los logs agregados
    # Asegurar que MapReduce sepa dónde levantar su servidor histórico
    "MAPRED-SITE.XML_mapreduce.jobhistory.address": f"{HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_MAPRED_JOB_HISTORY_PORT}",
    "MAPRED-SITE.XML_mapreduce.jobhistory.webapp.address": f"{HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_MAPRED_LOG_SERVER_PORT}",
}
with open("envs/common.env", "w") as f:
    for key, value in env_content.items():
        f.write(f"{key}={value}\n")

hadoop_compose_file_name = "hadoop-cluster.docker-compose.yml"
hadoop_compose_dict = {
    "name": "hadoop-cluster",
    "networks": {"hadoop-cluster": {"driver": "bridge"}},
    "services": {
        "namenode": {
            "image": "apache/hadoop:3.4.3",
            "container_name": "namenode",
            "command": [
                "bash",
                "-c",
                " ".join(
                    [
                        f"sudo chown -R $(id -u hadoop):$(id -g hadoop) {HADOOP_WORKDIR} &&",
                        f"sudo mkdir -p {HADOOP_NAMENODE_NAMEDIR} &&",
                        f"sudo chown -R $(id -u hadoop):$(id -g hadoop) {HADOOP_NAMENODE_NAMEDIR} &&",
                        f"if [ ! -d {HADOOP_NAMENODE_NAMEDIR}/current ]; then hdfs namenode -Ddfs.namenode.name.dir={HADOOP_NAMENODE_NAMEDIR} -format -force; fi &&",
                        "hdfs",
                        "namenode",
                        f"-Dfs.defaultFS=hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}",
                        # f"-Ddfs.replication={HADOOP_REPLICATION}",
                        # f"-Ddfs.namenode.name.dir={HADOOP_NAMENODE_NAMEDIR}",
                        # "-Ddfs.namenode.name.dir.perm=775",
                        # "-Ddfs.permissions.enabled=false",
                        f"-Ddfs.namenode.rpc-address=0.0.0.0:{HADOOP_NAMENODE_PORT}",
                        f"-Ddfs.namenode.http-address=0.0.0.0:{HADOOP_NAMENODE_WEBUI_PORT}",
                    ]
                ),
            ],
            "env_file": ["envs/common.env"],
            "volumes": [
                f"{os.path.join(DOCKER_MOUNTDIR,"namenode","work-dir")}:{HADOOP_WORKDIR}",
                f"{os.path.join(DOCKER_MOUNTDIR,"namenode","name-dir")}:{HADOOP_NAMENODE_NAMEDIR}",
            ],
            "networks": ["hadoop-cluster"],
            "hostname": HADOOP_NAMENODE_HOSTNAME,
            "ports": [
                f"{HADOOP_NAMENODE_PORT}:{HADOOP_NAMENODE_PORT}",
                f"{HADOOP_NAMENODE_WEBUI_PORT}:{HADOOP_NAMENODE_WEBUI_PORT}",
            ],
            "extra_hosts": [
                f"{DOCKER_INTERNAL_HOST}:host-gateway",
            ],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "1.0", "memory": "512M"}}},
        },
        "resourcemanager": {
            "image": "apache/hadoop:3.4.3",
            "container_name": "resourcemanager",
            "command": [
                "bash",
                "-c",
                " ".join(
                    [
                        f"sudo chown -R $(id -u hadoop):$(id -g hadoop) {HADOOP_WORKDIR} &&",
                        f"mapred --daemon start historyserver &&",
                        "yarn",
                        "resourcemanager",
                        # f"-Dfs.defaultFS=hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}",
                        # f"-Dyarn.resourcemanager.hostname={HADOOP_RESOURCEMANAGER_HOSTNAME}",
                        f"-Dyarn.resourcemanager.webapp.address=0.0.0.0:{HADOOP_RESOURCEMANAGER_WEBUI_PORT}",
                        f"-Dyarn.resourcemanager.address=0.0.0.0:{HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT}",
                        f"-Dyarn.resourcemanager.scheduler.address=0.0.0.0:{HADOOP_RESOURCEMANAGER_SCHEDULER_PORT}",
                        f"-Dyarn.resourcemanager.resource-tracker.address=0.0.0.0:{HADOOP_RESOURCEMANAGER_TRACKER_PORT}",
                        f"-Dyarn.resourcemanager.admin.address=0.0.0.0:{HADOOP_RESOURCEMANAGER_ADMIN_PORT}",
                    ]
                ),
            ],
            "env_file": ["envs/common.env"],
            "volumes": [
                f"{os.path.join(DOCKER_MOUNTDIR,"resourcemanager","work-dir")}:{HADOOP_WORKDIR}",
            ],
            "networks": ["hadoop-cluster"],
            "hostname": HADOOP_RESOURCEMANAGER_HOSTNAME,
            "ports": [
                f"{HADOOP_RESOURCEMANAGER_WEBUI_PORT}:{HADOOP_RESOURCEMANAGER_WEBUI_PORT}",
                f"{HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT}:{HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT}",
                f"{HADOOP_RESOURCEMANAGER_SCHEDULER_PORT}:{HADOOP_RESOURCEMANAGER_SCHEDULER_PORT}",
                f"{HADOOP_RESOURCEMANAGER_TRACKER_PORT}:{HADOOP_RESOURCEMANAGER_TRACKER_PORT}",
                f"{HADOOP_RESOURCEMANAGER_ADMIN_PORT}:{HADOOP_RESOURCEMANAGER_ADMIN_PORT}",
                f"{HADOOP_MAPRED_JOB_HISTORY_PORT}:{HADOOP_MAPRED_JOB_HISTORY_PORT}",
                f"{HADOOP_MAPRED_LOG_SERVER_PORT}:{HADOOP_MAPRED_LOG_SERVER_PORT}",
            ],
            "extra_hosts": [
                f"{DOCKER_INTERNAL_HOST}:host-gateway",
            ],
            "dns": DOCKER_DNS,
            "depends_on": {"namenode": {"condition": "service_started"}},
            "deploy": {"resources": {"limits": {"cpus": "2.0", "memory": "512M"}}},
        },
    },
}


for i in range(0, HADOOP_NUM_WORKERS):

    # Programmatically add DataNodes
    hadoop_compose_dict["services"][HADOOP_DATANODE_NAMES[i]] = {
        "image": "apache/hadoop:3.4.3",
        "container_name": HADOOP_DATANODE_NAMES[i],
        "command": [
            "bash",
            "-c",
            " ".join(
                [
                    f"sudo chown -R $(id -u hadoop):$(id -g hadoop) {HADOOP_WORKDIR} &&",
                    f"sudo mkdir -p {HADOOP_DATANODE_DATADIR} &&",
                    f"sudo chown -R $(id -u hadoop):$(id -g hadoop) {HADOOP_DATANODE_DATADIR} &&",
                    "hdfs",
                    "datanode",
                    # f"-Dfs.defaultFS=hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}",
                    # f"-Ddfs.datanode.data.dir={HADOOP_DATANODE_DATADIR}",
                    # "-Ddfs.datanode.data.dir.perm=775",
                    # "-Ddfs.permissions.enabled=false",
                    f"-Ddfs.datanode.address=0.0.0.0:{HADOOP_DATANODE_TRANSFER_PORTS[i]}",
                    f"-Ddfs.datanode.http.address=0.0.0.0:{HADOOP_DATANODE_WEBUI_PORTS[i]}",
                    f"-Ddfs.datanode.ipc.address=0.0.0.0:{HADOOP_DATANODE_IPC_PORTS[i]}",
                    # "-Ddfs.datanode.use.datanode.hostname=true",
                ]
            ),
        ],
        "env_file": ["envs/common.env"],
        "volumes": [
            f"{os.path.join(DOCKER_MOUNTDIR,HADOOP_DATANODE_NAMES[i],"work-dir")}:{HADOOP_WORKDIR}",
            f"{os.path.join(DOCKER_MOUNTDIR,HADOOP_DATANODE_NAMES[i],"data-dir")}:{HADOOP_DATANODE_DATADIR}",
        ],
        "networks": ["hadoop-cluster"],
        "hostname": HADOOP_DATANODE_HOSTNAMES[i],
        "ports": [
            f"{HADOOP_DATANODE_WEBUI_PORTS[i]}:{HADOOP_DATANODE_WEBUI_PORTS[i]}",
            f"{HADOOP_DATANODE_TRANSFER_PORTS[i]}:{HADOOP_DATANODE_TRANSFER_PORTS[i]}",
            f"{HADOOP_DATANODE_IPC_PORTS[i]}:{HADOOP_DATANODE_IPC_PORTS[i]}",
        ],
        "extra_hosts": [
            f"{DOCKER_INTERNAL_HOST}:host-gateway",
        ],
        "dns": DOCKER_DNS,
        "deploy": {"resources": {"limits": {"cpus": "1.0", "memory": "512M"}}},
        "depends_on": {
            "namenode": {"condition": "service_started"},
            "resourcemanager": {"condition": "service_started"},
        }
        | {
            HADOOP_DATANODE_NAMES[j]: {"condition": "service_started"}
            for j in range(0, i)
        },
    }

    # Programmatically add NodeManagers
    hadoop_compose_dict["services"][HADOOP_NODEMANAGER_NAMES[i]] = {
        "image": "apache/hadoop:3.4.3",
        "container_name": HADOOP_NODEMANAGER_NAMES[i],
        "command": [
            "bash",
            "-c",
            " ".join(
                [
                    f"sudo chown -R $(id -u hadoop):$(id -g hadoop) {HADOOP_WORKDIR} &&",
                    "yarn",
                    "nodemanager",
                    # f"-Dyarn.resourcemanager.hostname={HADOOP_RESOURCEMANAGER_HOSTNAME}",
                    # f"-Dyarn.nodemanager.aux-services=mapreduce_shuffle",
                    f"-Dyarn.resourcemanager.resource-tracker.address={HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_RESOURCEMANAGER_TRACKER_PORT}",
                    f"-Dyarn.nodemanager.address=0.0.0.0:{HADOOP_NODEMANAGER_RPC_PORTS[i]}",
                    f"-Dyarn.nodemanager.webapp.address=0.0.0.0:{HADOOP_NODEMANAGER_WEBUI_PORTS[i]}",
                ]
            ),
        ],
        "env_file": ["envs/common.env"],
        "volumes": [
            f"{os.path.join(DOCKER_MOUNTDIR,HADOOP_NODEMANAGER_NAMES[i],"work-dir")}:{HADOOP_WORKDIR}",
        ],
        "networks": ["hadoop-cluster"],
        "hostname": HADOOP_NODEMANAGER_HOSTNAMES[i],
        "ports": [
            f"{HADOOP_NODEMANAGER_WEBUI_PORTS[i]}:{HADOOP_NODEMANAGER_WEBUI_PORTS[i]}",
            f"{HADOOP_NODEMANAGER_RPC_PORTS[i]}:{HADOOP_NODEMANAGER_RPC_PORTS[i]}",
        ],
        "extra_hosts": [
            f"{DOCKER_INTERNAL_HOST}:host-gateway",
        ],
        "dns": DOCKER_DNS,
        "deploy": {"resources": {"limits": {"cpus": "3.0", "memory": "1792M"}}},
        "depends_on": {
            "namenode": {"condition": "service_started"},
            "resourcemanager": {"condition": "service_started"},
        }
        | {
            HADOOP_DATANODE_NAMES[j]: {"condition": "service_started"}
            for j in range(HADOOP_NUM_WORKERS)
        }
        | {
            HADOOP_NODEMANAGER_NAMES[j]: {"condition": "service_started"}
            for j in range(0, i)
        },
    }


hadoop_compose_yaml_contents = yaml.dump(
    hadoop_compose_dict, default_flow_style=False, sort_keys=False, indent=4
)
with open(
    os.path.join(LOCALHOST_WORKDIR, hadoop_compose_file_name), "w"
) as hadoop_compose_file:
    hadoop_compose_file.write(hadoop_compose_yaml_contents)
print(
    f"Successfully created: {os.path.relpath(os.path.join(LOCALHOST_WORKDIR,hadoop_compose_file_name))}"
)
display(Markdown(f"```yaml\n{hadoop_compose_yaml_contents}\n```"))

In [ ]:
!docker compose -f hadoop-cluster.docker-compose.yml up -d --wait